# Polymarket Quant Playbook - Walk-Forward Backtest

Backtests all 6 formulas using walk-forward methodology with zero lookahead bias.

**Sections:**
1. Setup
2. Load dataset
3. EV Gap strategy
4. Kelly fraction sensitivity
5. KL divergence analysis
6. Bayesian updater accuracy
7. Summary dashboard
8. Next steps

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.stats import entropy
import warnings
warnings.filterwarnings('ignore')

from formulas.kelly_criterion import kelly, fractional_kelly
from formulas.ev_gap import compute_ev, scan_markets
from formulas.kl_divergence import binary_kl
from formulas.bayesian_update import bayesian_update, sequential_update

plt.style.use('dark_background')
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': 0.2})
print("Ready")


## 2. Load Dataset

In [ ]:
df = pd.read_csv('../data/sample_markets.csv')
print(f"Markets : {len(df)}")
print(f"Win rate : {df['outcome'].mean():.1%}")
df.head(8)


## 3. EV Gap Strategy

In [ ]:
FEE          = 0.02
EV_THRESHOLD = 0.05
KELLY_FRAC   = 0.50
BANKROLL_0   = 1000.0
N_DAYS       = int(df['day'].max()) + 1

df['ev']      = df.apply(lambda r: compute_ev(r['model_p'], r['price'], FEE), axis=1)
df['kelly_f'] = df.apply(lambda r: fractional_kelly(r['model_p'], r['price'], KELLY_FRAC), axis=1)
df['bet']     = df['ev'] > EV_THRESHOLD

bets = df[df['bet']].copy()
print(f"Bets placed    : {len(bets)} / {len(df)}  ({len(bets)/len(df):.0%})")
print(f"Avg EV on bets : {bets['ev'].mean():.4f}")
print(f"Win rate       : {bets['outcome'].mean():.1%}")


In [ ]:
bankroll = BANKROLL_0
br_hist  = []
pnl_hist = []

for day in range(N_DAYS):
    day_bets = bets[bets['day'] == day]
    day_pnl  = 0.0
    for _, row in day_bets.iterrows():
        bet_size = row['kelly_f'] * bankroll
        payout   = (1 / row['price']) - 1
        day_pnl += bet_size * payout if row['outcome'] == 1 else -bet_size
    bankroll = max(bankroll + day_pnl, 0.01)
    br_hist.append(bankroll)
    pnl_hist.append(day_pnl)

br_series  = pd.Series(br_hist)
pnl_series = pd.Series(pnl_hist)

roi    = (br_series.iloc[-1] - BANKROLL_0) / BANKROLL_0
sharpe = pnl_series.mean() / (pnl_series.std() + 1e-9) * np.sqrt(252)
max_dd = ((br_series.cummax() - br_series) / br_series.cummax()).max()

print(f"Starting : ${BANKROLL_0:,.0f}")
print(f"Final    : ${br_series.iloc[-1]:,.2f}")
print(f"ROI      : {roi:+.1%}")
print(f"Sharpe   : {sharpe:.2f}")
print(f"Max DD   : {max_dd:.1%}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(br_series, color='#00c896', linewidth=2)
ax1.axhline(BANKROLL_0, color='gray', linestyle='--', alpha=0.5)
ax1.fill_between(range(N_DAYS), BANKROLL_0, br_series,
                 where=br_series >= BANKROLL_0, alpha=0.15, color='#00c896')
ax1.fill_between(range(N_DAYS), BANKROLL_0, br_series,
                 where=br_series < BANKROLL_0, alpha=0.15, color='crimson')
ax1.set_title(f'Bankroll  |  ROI {roi:+.1%}  Sharpe {sharpe:.2f}  Max DD {max_dd:.1%}')
ax1.set_xlabel('Day')
ax1.set_ylabel('USD')

ax2.bar(range(N_DAYS), pnl_series,
        color=['#00c896' if p >= 0 else 'crimson' for p in pnl_series], alpha=0.8)
ax2.axhline(0, color='white', linewidth=0.8)
ax2.set_title('Daily P&L')
ax2.set_xlabel('Day')
ax2.set_ylabel('USD')

plt.tight_layout()
plt.show()


## 4. Kelly Fraction Sensitivity

In [ ]:
fractions = [0.10, 0.25, 0.50, 0.75, 1.00]
colors    = ['#ff6b6b', '#ffa94d', '#00c896', '#4fa8d5', '#b197fc']
results   = {}

for frac in fractions:
    br   = BANKROLL_0
    hist = []
    for day in range(N_DAYS):
        for _, row in bets[bets['day'] == day].iterrows():
            f      = max(0, kelly(row['model_p'], row['price'])) * frac
            bs     = f * br
            payout = (1 / row['price']) - 1
            br     = max(br + (bs * payout if row['outcome'] == 1 else -bs), 0.01)
        hist.append(br)
    results[frac] = hist

plt.figure(figsize=(12, 5))
for (frac, hist), c in zip(results.items(), colors):
    plt.plot(hist, label=f'{frac}x Kelly  final=${hist[-1]:,.0f}', color=c, linewidth=2)
plt.axhline(BANKROLL_0, color='white', linestyle='--', alpha=0.4)
plt.yscale('log')
plt.title('Bankroll Growth by Kelly Fraction')
plt.xlabel('Day')
plt.ylabel('USD (log scale)')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()


## 5. KL Divergence Analysis

In [ ]:
pairs = [(df.iloc[i]['market_id'], df.iloc[i+1]['market_id'])
         for i in range(0, min(20, len(df)-1), 2)]

pair_rows = []
for a, b in pairs:
    ra = df[df['market_id'] == a].iloc[0]
    rb = df[df['market_id'] == b].iloc[0]
    kl = binary_kl(ra['price'], rb['price'])
    pair_rows.append({
        'pair':    f"{a} / {b}",
        'price_a': ra['price'],
        'price_b': rb['price'],
        'kl':      round(kl, 4),
        'arb':     kl > 0.2,
    })

pair_df = pd.DataFrame(pair_rows)
print(f"Pairs scanned : {len(pair_df)}")
print(f"Arb signals   : {pair_df['arb'].sum()}")
print()
print(pair_df.to_string(index=False))


In [ ]:
plt.figure(figsize=(10, 4))
bar_colors = ['#00c896' if arb else '#555' for arb in pair_df['arb']]
plt.bar(range(len(pair_df)), pair_df['kl'], color=bar_colors, alpha=0.85)
plt.axhline(0.2, color='crimson', linestyle='--', linewidth=2, label='KL threshold = 0.2')
plt.xticks(range(len(pair_df)), pair_df['pair'], rotation=45, ha='right', fontsize=7)
plt.ylabel('KL Score')
plt.title('KL Divergence by Market Pair')
plt.legend()
plt.tight_layout()
plt.show()


## 6. Bayesian Updater Accuracy

In [ ]:
np.random.seed(42)
static_errors  = []
updated_errors = []
prior          = 0.50

for _, row in df.iterrows():
    outcome = row['outcome']
    stream  = []
    for _ in range(3):
        lh_t = np.random.uniform(0.60, 0.85) if outcome == 1 else np.random.uniform(0.25, 0.50)
        lh_f = np.clip(1 - lh_t + np.random.uniform(-0.05, 0.05), 0.1, 0.9)
        stream.append((lh_t, lh_f))

    posteriors = sequential_update(prior, stream)
    final      = posteriors[-1]

    static_errors.append((prior - outcome) ** 2)
    updated_errors.append((final - outcome) ** 2)

static_mse  = np.mean(static_errors)
updated_mse = np.mean(updated_errors)
improvement = (static_mse - updated_mse) / static_mse

print(f"Static MSE   : {static_mse:.4f}")
print(f"Bayesian MSE : {updated_mse:.4f}")
print(f"Gain         : +{improvement:.1%}")


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.hist(static_errors,  bins=15, alpha=0.6, color='crimson',  label='Static', edgecolor='white', lw=0.3)
ax1.hist(updated_errors, bins=15, alpha=0.6, color='#00c896', label='Bayesian', edgecolor='white', lw=0.3)
ax1.set_title('Prediction Error Distribution')
ax1.set_xlabel('Squared Error')
ax1.legend()

lim = max(max(static_errors), max(updated_errors))
ax2.scatter(static_errors, updated_errors, alpha=0.6, color='#4fa8d5', s=40)
ax2.plot([0, lim], [0, lim], 'r--', alpha=0.5, label='No improvement')
ax2.set_title('Static vs Bayesian per Market')
ax2.set_xlabel('Static Error')
ax2.set_ylabel('Bayesian Error')
ax2.legend()

plt.tight_layout()
plt.show()


## 7. Summary Dashboard

In [ ]:
fig = plt.figure(figsize=(16, 8))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(br_series, color='#00c896', linewidth=2.5)
ax1.fill_between(range(N_DAYS), BANKROLL_0, br_series,
                 where=br_series >= BANKROLL_0, alpha=0.15, color='#00c896')
ax1.axhline(BANKROLL_0, color='white', linestyle='--', alpha=0.4)
ax1.set_title(f'Bankroll  |  ROI {roi:+.1%}  Sharpe {sharpe:.2f}  Max DD {max_dd:.1%}')
ax1.set_xlabel('Day')
ax1.set_ylabel('USD')

ax2 = fig.add_subplot(gs[0, 2])
ax2.hist(df['ev'], bins=20, color='#4fa8d5', alpha=0.8, edgecolor='white', lw=0.3)
ax2.axvline(EV_THRESHOLD, color='crimson', linestyle='--', lw=2, label='Threshold')
ax2.set_title(f'EV Distribution ({len(bets)} bets)')
ax2.set_xlabel('Net EV')
ax2.legend(fontsize=8)

ax3 = fig.add_subplot(gs[1, 0])
finals   = [v[-1] for v in results.values()]
bars     = ax3.bar([f'{f}x' for f in fractions], finals, color=colors, alpha=0.85)
ax3.axhline(BANKROLL_0, color='white', linestyle='--', alpha=0.4)
ax3.set_title('Final Bankroll by Kelly Fraction')
ax3.set_ylabel('USD')
for bar, val in zip(bars, finals):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             f'${val:,.0f}', ha='center', fontsize=8)

ax4 = fig.add_subplot(gs[1, 1])
c4  = ['#00c896' if arb else '#555' for arb in pair_df['arb']]
ax4.bar(range(len(pair_df)), pair_df['kl'], color=c4, alpha=0.85)
ax4.axhline(0.2, color='crimson', linestyle='--', lw=2)
ax4.set_title(f'KL Scores ({pair_df["arb"].sum()} arb signals)')
ax4.set_xlabel('Pair index')
ax4.set_ylabel('KL Score')

ax5 = fig.add_subplot(gs[1, 2])
b5  = ax5.bar(['Static', 'Bayesian'], [static_mse, updated_mse],
              color=['crimson', '#00c896'], alpha=0.85, width=0.5)
ax5.set_title(f'Prediction MSE  (+{improvement:.1%} gain)')
ax5.set_ylabel('Mean Squared Error')
for bar, val in zip(b5, [static_mse, updated_mse]):
    ax5.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', fontsize=10)

plt.suptitle('Polymarket Quant Playbook - Backtest Summary', fontsize=15, fontweight='bold')
os.makedirs('../data', exist_ok=True)
plt.savefig('../data/backtest_summary.png', dpi=120, bbox_inches='tight')
plt.show()
print("Saved to data/backtest_summary.png")


## 8. Next Steps

| Step | Action |
|------|--------|
| Real data | Replace CSV with Polymarket API via bot/data_fetcher.py |
| model_p | Build a real model using polls, sentiment, news NLP |
| Walk-forward | 60-day train, 30-day test rolling window |
| Paper trade | Run bot/main.py --loop 60 for 2 weeks, no real money |
| Go live | Raise EV threshold to 0.08+, monitor Sharpe daily |

**Risk checklist before going live:**
- Sharpe > 1.5 on out-of-sample data
- Max drawdown < 20% in all test windows
- Edge persists in last 30 days (no decay)
- Kelly fraction set to 0.5x or lower
- Fee sensitivity tested at 3%
